# OT metric forensics in CNT primitives and field; DINO/SAM rows of `tab:app-representation`

Printed output only. `item2_representation_n40.json`, which this notebook was meant to write, is not in the archive.

**Needs:** DTD, the CNT checkpoint, SAM checkpoint.

**Maintained runnable path:** none. This notebook (with `never_gonna_give_you_up.ipynb`, in this same directory) is the only record of `tab:app-representation`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch, numpy as np, random, json
import torch.nn.functional as F
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root    = os.path.join(PROJECT_DIR, "dtd/images")
OUT         = os.path.join(PROJECT_DIR, "ot_dualmetric"); os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = 256
N_ETA = 21; etas = np.linspace(0,1,N_ETA)
N_SUB = 512
N_PAIRS = 40
print("device:", device)

In [ ]:
!pip install -q timm pot lpips
import timm, ot as pot
import torchvision.models as tvm
import lpips

_m = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(device)
_s = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(device)

dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).eval().to(device)
for p in dino.parameters(): p.requires_grad=False
def feat_dino(img):
    x = F.interpolate((img+1)/2, size=(518,518), mode='bilinear', align_corners=False); x=(x-_m)/_s
    with torch.no_grad():
        f = dino.forward_features(x)
        f = f['x_norm_patchtokens'] if isinstance(f,dict) and 'x_norm_patchtokens' in f else f
    return f.squeeze(0)
def dino_map_layer(img, block=8):
    x = F.interpolate((img+1)/2, size=(518,518), mode='bilinear', align_corners=False); x=(x-_m)/_s
    return dino.get_intermediate_layers(x, n=[block], reshape=False)[0]   # (1,N,D)
FEAT = lambda im: dino_map_layer(im, block=8)   # the invertible feature used for decoding

!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
from sam2.build_sam import build_sam2
from huggingface_hub import hf_hub_download
_ck = hf_hub_download("facebook/sam2-hiera-small", "sam2_hiera_small.pt")
sam2 = build_sam2("sam2_hiera_s.yaml", _ck, device=device).eval()
def feat_sam(img):
    x = F.interpolate((img+1)/2, size=(1024,1024), mode='bilinear', align_corners=False); x=(x-_m)/_s
    with torch.no_grad():
        out = sam2.image_encoder(x)
        f = out['vision_features'] if isinstance(out,dict) else out
    return f.squeeze(0).flatten(1).t()

_vgg = tvm.vgg16(weights='IMAGENET1K_V1').features[:23].eval().to(device)
for p in _vgg.parameters(): p.requires_grad=False
def feat_vgg(img):
    x=(img+1)/2; x=(x-_m)/_s
    with torch.no_grad(): f=_vgg(x)
    return f.squeeze(0).flatten(1).t()
lpips_fn = lpips.LPIPS(net='vgg').to(device).eval()
print("extractors + judges ready")

In [ ]:
from PIL import Image
def load_img(path):
    pil = Image.open(path).convert('RGB').resize((IMG,IMG))
    return (torch.from_numpy(np.asarray(pil)).float()/127.5-1).permute(2,0,1).unsqueeze(0).to(device)
def pick_img(cat, seed):
    random.seed(seed); f=random.choice(os.listdir(os.path.join(dtd_root,cat)))
    return load_img(os.path.join(dtd_root,cat,f))

def ot_match(Fa, Fb):
    A=Fa.double().cpu().numpy(); B=Fb.double().cpu().numpy()
    C=pot.dist(A,B,metric='sqeuclidean'); C/=C.max()+1e-12
    G=pot.emd(np.full(len(A),1/len(A)), np.full(len(B),1/len(B)), C)
    tgt=(G@B)/(G.sum(1,keepdims=True)+1e-12)
    return torch.tensor(tgt, device=Fa.device, dtype=Fa.dtype)

def interp_feature(Fa, Fb, eta, mode):    # Fa,Fb: (1,N,D)
    a,b = Fa[0], Fb[0]
    if mode=='ot':
        tgt=ot_match(a,b); return ((1-eta)*a+eta*tgt).unsqueeze(0)
    return ((1-eta)*a+eta*b).unsqueeze(0)

def tv(x): return (x[...,1:,:]-x[...,:-1,:]).abs().mean()+(x[...,:,1:]-x[...,:,:-1]).abs().mean()
def invert_to_image(target_feat, feat_fn, steps=600, lr=0.02, tv_w=0.05, init=None):
    img=(torch.randn(1,3,IMG,IMG,device=device)*0.1).requires_grad_(True) if init is None \
        else init.clone().detach().requires_grad_(True)
    opt=torch.optim.Adam([img],lr=lr); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,steps)
    for _ in range(steps):
        x=torch.tanh(img)
        loss=(feat_fn(x)-target_feat).pow(2).mean()+tv_w*tv(x)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return torch.tanh(img).detach()
print("interp + inversion ready")

In [ ]:
def subsamp(X, n, seed=0):
    g=torch.Generator(device=X.device).manual_seed(seed)
    idx=torch.randperm(X.shape[0], generator=g, device=X.device)[:n]; return X[idx]
def cloud_spread(P):
    c=P.mean(0,keepdim=True); return (P-c).pow(2).sum(1).mean().item()

random.seed(0); cats=sorted(os.listdir(dtd_root))
sample_pairs=[tuple(random.sample(cats,2)) for _ in range(N_PAIRS)]

SPREAD_DIR=os.path.join(OUT,'spread'); os.makedirs(SPREAD_DIR,exist_ok=True)
EXTR={'dino':feat_dino,'sam':feat_sam}
rows_spread=[]
for i,(a,b) in enumerate(sample_pairs):
    tag=f"{a}__{b}__{i}"; fp=os.path.join(SPREAD_DIR,tag+'.json')
    if os.path.exists(fp): rows_spread.append(json.load(open(fp))); continue
    rec={'pair':f"{a}/{b}"}
    ia,ib = pick_img(a,42+i), pick_img(b,99+i)
    for name,fn in EXTR.items():
        A=subsamp(fn(ia).detach(),N_SUB,i); B=subsamp(fn(ib).detach(),N_SUB,i)
        sA,sB=cloud_spread(A),cloud_spread(B); avg=0.5*(sA+sB)
        sL=cloud_spread(0.5*A+0.5*B)
        sO=cloud_spread(0.5*A+0.5*ot_match(A,B))
        rec[f'{name}_lin']=sL/avg; rec[f'{name}_ot']=sO/avg
    json.dump(rec,open(fp,'w')); rows_spread.append(rec)
    if i%5==0: print(f"[{i+1}/{N_PAIRS}] dino LIN {rec['dino_lin']:.2f} OT {rec['dino_ot']:.2f} | sam LIN {rec['sam_lin']:.2f} OT {rec['sam_ot']:.2f}")
print("spread sweep done")

In [ ]:
def mean_ci(x):
    x=np.array(x); m=x.mean(); ci=1.96*x.std(ddof=1)/np.sqrt(len(x)); return m,ci
print(f"{'repr':>6} | LIN keeps      | OT keeps       | OT gain | n")
print('-'*60)
for name in ['dino','sam']:
    lin=[r[f'{name}_lin'] for r in rows_spread]; ot=[r[f'{name}_ot'] for r in rows_spread]
    ml,cl=mean_ci(lin); mo,co=mean_ci(ot)
    print(f"{name:>6} | {ml:5.0%} ± {cl:4.0%}   | {mo:5.0%} ± {co:4.0%}   | {mo-ml:+5.0%}  | {len(lin)}")
print("\n(reference from prior CNT runs: texton 60%->87%, pixel 67%->99%)")

In [ ]:
import glob, os
VGG_PAIRS = sample_pairs[:40]          # reuse the first 10 from the spread sweep (same textures)
INV_STEPS = 600
VGG_DIR = os.path.join(OUT, 'vgg_inv'); os.makedirs(VGG_DIR, exist_ok=True)
cached = sorted(glob.glob(os.path.join(VGG_DIR, '*.json')))
print(f"cached pairs: {len(cached)} / 40   →  {40 - len(cached)} left")
print("last few:", [os.path.basename(c) for c in cached[-3:]])

In [ ]:
VGG_PAIRS = sample_pairs[:40]          # reuse the first 10 from the spread sweep (same textures)
INV_STEPS = 600
VGG_DIR = os.path.join(OUT, 'vgg_inv'); os.makedirs(VGG_DIR, exist_ok=True)

def r2_diag(d):
    d=np.array(d); d=d/(d[-1]+1e-12)
    return 1-((d-etas)**2).sum()/(((etas-etas.mean())**2).sum()+1e-9), d

def judge(img, img_a):
    with torch.no_grad():
        dv = (feat_vgg(img)-feat_vgg(img_a)).pow(2).mean().item()
        dl = lpips_fn(img.clamp(-1,1), img_a.clamp(-1,1)).item()
    return dv, dl

rows_vgg=[]
for i,(a,b) in enumerate(VGG_PAIRS):
    tag=f"{a}__{b}__{i}"; fp=os.path.join(VGG_DIR,tag+'.json')
    if os.path.exists(fp): rows_vgg.append(json.load(open(fp))); continue
    ia, ib = pick_img(a,42+i), pick_img(b,99+i)
    Fa, Fb = FEAT(ia).detach(), FEAT(ib).detach()        # block-8 (invertible) features
    img_a = invert_to_image(Fa, FEAT, steps=INV_STEPS)   # anchor: decoded A
    rec={'pair':f"{a}/{b}"}
    for mode in ['ot','linear']:
        prev=img_a; cv=[]; cl=[]
        for e in etas:
            feat = interp_feature(Fa, Fb, float(e), mode)
            img  = invert_to_image(feat, FEAT, steps=INV_STEPS, init=prev)  # warm-start
            prev = img
            dv,dl = judge(img, img_a)
            cv.append(dv); cl.append(dl)
        rec[f'{mode}_vgg_r2'], _ = r2_diag(cv)
        rec[f'{mode}_lpips_r2'], _ = r2_diag(cl)
        rec[f'{mode}_vgg_mid'] = (np.array(cv)/(cv[-1]+1e-12))[N_ETA//2]   # normalized midpoint
    json.dump(rec, open(fp,'w')); rows_vgg.append(rec)
    print(f"[{i+1}/10] {rec['pair']:>22}  VGG R²: ot={rec['ot_vgg_r2']:.2f} lin={rec['linear_vgg_r2']:.2f}"
          f" | LPIPS R²: ot={rec['ot_lpips_r2']:.2f} lin={rec['linear_lpips_r2']:.2f}")
print("VGG-via-inversion sweep done")

In [ ]:
from scipy.stats import wilcoxon
def summarize(key):
    ot =np.array([r[f'ot_{key}_r2'] for r in rows_vgg])
    lin=np.array([r[f'linear_{key}_r2'] for r in rows_vgg])
    print(f"{key.upper():>6} path R² (closer to 1 = straighter, better):")
    print(f"        OT  {ot.mean():.3f} ± {1.96*ot.std(ddof=1)/np.sqrt(len(ot)):.3f}")
    print(f"        LIN {lin.mean():.3f} ± {1.96*lin.std(ddof=1)/np.sqrt(len(lin)):.3f}")
    print(f"        OT wins {(ot>lin).sum()}/{len(ot)}", end="")
    try:
        p=wilcoxon(ot,lin,alternative='greater').pvalue; print(f" | Wilcoxon p={p:.3f}")
    except Exception as e: print(f" | wilcoxon: {e}")

for k in ['vgg','lpips']: summarize(k); print()

# midpoint placement (OT should sit closer to 0.5 if it spreads change evenly)
ot_mid =np.abs(np.array([r['ot_vgg_mid'] for r in rows_vgg])-0.5)
lin_mid=np.abs(np.array([r['linear_vgg_mid'] for r in rows_vgg])-0.5)
print(f"VGG |midpoint-0.5|: OT {ot_mid.mean():.3f}  LIN {lin_mid.mean():.3f}  (lower=better) | OT wins {(ot_mid<lin_mid).sum()}/{len(ot_mid)}")

In [ ]:
import json, glob
res_item2 = {
    'n': 40,
    'vgg_r2':   {'ot':0.842,'lin':0.821,'wins':'26/40','p':0.064},
    'lpips_r2': {'ot':0.825,'lin':0.759,'wins':'34/40','p':0.000},
    'vgg_midpoint': {'ot':0.071,'lin':0.127,'wins':'30/40'},
    'reading':'OT advantage significant on LPIPS (independent judge), NS on VGG path-linearity. '
              'Effect survives in DINO features -> not a CNT/VGG artifact. Closes item 2.'
}
json.dump(res_item2, open('/content/drive/MyDrive/DATA_ROOT/item2_representation_n40.json','w'), indent=2)
print("saved")

In [ ]:
import matplotlib.pyplot as plt

# Re-invert a few pairs and show the eta interpolation path (OT vs linear)
def show_inversion_path(pair_idx, n_show=5):
    a, b = sample_pairs[pair_idx]
    ia, ib = pick_img(a, 42+pair_idx), pick_img(b, 99+pair_idx)
    Fa, Fb = FEAT(ia).detach(), FEAT(ib).detach()

    show_etas = np.linspace(0, 1, n_show)
    fig, axes = plt.subplots(2, n_show+2, figsize=(2.1*(n_show+2), 4.6))

    def to_img(t): return np.clip((t[0].permute(1,2,0).cpu().numpy()+1)/2, 0, 1)

    # anchor images (real A, real B) in the outer columns
    for row in range(2):
        axes[row,0].imshow(to_img(ia)); axes[row,0].set_title('A' if row==0 else '', fontsize=9); axes[row,0].axis('off')
        axes[row,-1].imshow(to_img(ib)); axes[row,-1].set_title('B' if row==0 else '', fontsize=9); axes[row,-1].axis('off')

    for row, mode in enumerate(['ot', 'linear']):
        prev = invert_to_image(Fa, FEAT, steps=INV_STEPS)
        for k, e in enumerate(show_etas):
            feat = interp_feature(Fa, Fb, float(e), mode)
            img  = invert_to_image(feat, FEAT, steps=INV_STEPS, init=prev)
            prev = img
            axes[row, k+1].imshow(to_img(img)); axes[row, k+1].axis('off')
            if row == 0: axes[row, k+1].set_title(f'η={e:.2f}', fontsize=8)
        axes[row, 1].set_ylabel(mode.upper(), fontsize=11, rotation=0, labelpad=25, va='center')

    fig.suptitle(f"{a} → {b}   (top: OT, bottom: linear)", fontsize=11)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/DATA_ROOT/invpath_{pair_idx}.png', dpi=120, bbox_inches='tight')
    plt.show()

# clear wins for OT, plus #18 where LINEAR won (honesty)
for idx in [16, 6, 18]:      # spiralled/cobwebbed (OT big), an early one, knitted/striped (lin won LPIPS)
    show_inversion_path(idx)